# Day 14 — CuTe DSL Single-Stage SIMT GEMM

**Goal:** assemble everything from days 10–13 into a real (if naive)
SGEMM: `C = A @ B^T` for FP32 column-major operands, with a real MMA
atom, real shared memory, and a real K-mainloop. Single-stage — no
software pipelining, no producer-consumer split.

Four phases:

| Phase    | What                                                       |
|----------|------------------------------------------------------------|
| Prologue | Allocate smem, partition tiled-copies and tiled-MMA.       |
| Mainloop | For each K-tile: `gmem → smem`, `smem → rmem`, `mma`.       |
| Sync     | `barrier()` so all threads see the new smem tile.          |
| Epilogue | Write the per-thread accumulator out to `C`.               |

New primitives this day introduces:
- `cute.local_tile` — slice a per-CTA `(BLK_M, BLK_K, k)` view
- `cutlass.utils.SmemAllocator` — shared-memory arena
- `cute.nvgpu.MmaUniversalOp` + `make_tiled_mma` — SIMT FMA MMA
- `thr_mma.partition_A/B/C` — per-thread MMA operand views
- `cute.gemm` — the inner K-block MMA
- `cute.autovec_copy` — smem→rmem with auto-vectorization
- `cute.arch.cp_async_*` — async gmem→smem copies


In [ ]:
from typing import Tuple
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch

cutlass.cuda.initialize_cuda_context()


## Host: SmemAllocator-friendly layouts + tiled copies + tiled MMA

Filled in for you. The kernel below has three TODOs (A → B → C) — tiling
the global tensors, partitioning smem and MMA fragments, and the
K-mainloop.


In [ ]:
class SimpleSGemm:
    def __init__(self, cta_tiler=(64, 64, 8), num_threads=128):
        self.cta_tiler = cta_tiler
        self.bM, self.bN, self.bK = cta_tiler
        self.num_threads = num_threads

    @cute.jit
    def __call__(self, mA, mB, mC):
        sA_layout = cute.make_layout((self.bM, self.bK), stride=(1, self.bM))
        sB_layout = cute.make_layout((self.bN, self.bK), stride=(1, self.bN))

        num_vec: cutlass.Constexpr = 4
        def make_g2s(tiler_lead, dtype):
            t = cute.make_layout((tiler_lead, self.num_threads // tiler_lead),
                                  stride=(1, tiler_lead))
            v = cute.make_layout((num_vec, 1))
            atom = cute.make_copy_atom(cute.nvgpu.cpasync.CopyG2SOp(),
                                        dtype,
                                        num_bits_per_copy=dtype.width * num_vec)
            return cute.make_tiled_copy_tv(atom, t, v)
        tiled_copy_A = make_g2s(self.bM // num_vec, mA.element_type)
        tiled_copy_B = make_g2s(self.bN // num_vec, mB.element_type)

        op = cute.nvgpu.MmaUniversalOp(cutlass.Float32)
        atoms_layout = cute.make_layout((self.num_threads // 16, 16, 1),
                                         stride=(16, 1, 0))
        permutation = (cute.make_layout((atoms_layout.shape[0], 4), stride=(4, 1)),
                       cute.make_layout((atoms_layout.shape[1], 4), stride=(4, 1)),
                       None)
        tiled_mma = cute.make_tiled_mma(op, atoms_layout, permutation_mnk=permutation)

        grid = (*cute.ceil_div(mC.shape, (self.bM, self.bN)), 1)
        self.kernel(mA, mB, mC, sA_layout, sB_layout,
                    tiled_copy_A, tiled_copy_B, tiled_mma).launch(
            grid=grid, block=(cute.size(atoms_layout), 1, 1),
        )

    @cute.kernel
    def kernel(self, mA, mB, mC, sA_layout, sB_layout,
               tiled_copy_A, tiled_copy_B, tiled_mma):
        tidx, _, _ = cute.arch.thread_idx()
        bidx, bidy, _ = cute.arch.block_idx()
        thr_mma = tiled_mma.get_slice(tidx)

        # TODO (A): slice global tensors into per-CTA tiles via cute.local_tile.
        # TODO (B): SmemAllocator + partition_S/D for the tiled copies +
        #           partition_A/B/C for the MMA + accumulator fragment fill(0).
        # TODO (C): mainloop — cp.async gmem→smem, barrier, smem→rmem
        #           autovec_copy, cute.gemm, barrier.  Then epilogue copy.
        raise NotImplementedError


# M, N, K = 256, 256, 64
# a = torch.empty(K, M, dtype=torch.float32).normal_().permute(1, 0).cuda()
# b = torch.empty(K, N, dtype=torch.float32).normal_().permute(1, 0).cuda()
# c = torch.empty(N, M, dtype=torch.float32).zero_().permute(1, 0).cuda()
# SimpleSGemm()(from_dlpack(a, assumed_align=16),
#               from_dlpack(b, assumed_align=16),
#               from_dlpack(c, assumed_align=16))
# torch.cuda.synchronize()
# torch.testing.assert_close(c.cpu(), torch.einsum("mk,nk->mn", a, b).cpu(),
#                            atol=1e-3, rtol=1e-4); print("OK")


---

## Reference solution

Overwrites the kernel above with the complete implementation, then runs
one verification.


In [ ]:
class SimpleSGemm:
    def __init__(self, cta_tiler=(64, 64, 8), num_threads=128):
        self.cta_tiler = cta_tiler
        self.bM, self.bN, self.bK = cta_tiler
        self.num_threads = num_threads

    @cute.jit
    def __call__(self, mA, mB, mC):
        sA_layout = cute.make_layout((self.bM, self.bK), stride=(1, self.bM))
        sB_layout = cute.make_layout((self.bN, self.bK), stride=(1, self.bN))

        num_vec: cutlass.Constexpr = 4
        def make_g2s(tiler_lead, dtype):
            t = cute.make_layout((tiler_lead, self.num_threads // tiler_lead),
                                  stride=(1, tiler_lead))
            v = cute.make_layout((num_vec, 1))
            atom = cute.make_copy_atom(cute.nvgpu.cpasync.CopyG2SOp(),
                                        dtype,
                                        num_bits_per_copy=dtype.width * num_vec)
            return cute.make_tiled_copy_tv(atom, t, v)
        tiled_copy_A = make_g2s(self.bM // num_vec, mA.element_type)
        tiled_copy_B = make_g2s(self.bN // num_vec, mB.element_type)

        op = cute.nvgpu.MmaUniversalOp(cutlass.Float32)
        atoms_layout = cute.make_layout((self.num_threads // 16, 16, 1),
                                         stride=(16, 1, 0))
        permutation = (cute.make_layout((atoms_layout.shape[0], 4), stride=(4, 1)),
                       cute.make_layout((atoms_layout.shape[1], 4), stride=(4, 1)),
                       None)
        tiled_mma = cute.make_tiled_mma(op, atoms_layout, permutation_mnk=permutation)

        grid = (*cute.ceil_div(mC.shape, (self.bM, self.bN)), 1)
        self.kernel(mA, mB, mC, sA_layout, sB_layout,
                    tiled_copy_A, tiled_copy_B, tiled_mma).launch(
            grid=grid, block=(cute.size(atoms_layout), 1, 1),
        )

    @cute.kernel
    def kernel(self, mA, mB, mC, sA_layout, sB_layout,
               tiled_copy_A, tiled_copy_B, tiled_mma):
        tidx, _, _ = cute.arch.thread_idx()
        bidx, bidy, _ = cute.arch.block_idx()
        thr_mma = tiled_mma.get_slice(tidx)

        # (A) slice global tensors
        coord = (bidx, bidy, None)
        gA = cute.local_tile(mA, tiler=self.cta_tiler, coord=coord, proj=(1, None, 1))
        gB = cute.local_tile(mB, tiler=self.cta_tiler, coord=coord, proj=(None, 1, 1))
        gC = cute.local_tile(mC, tiler=self.cta_tiler, coord=coord, proj=(1, 1, None))

        # (B) smem + partitioning
        smem = cutlass.utils.SmemAllocator()
        sA = smem.allocate_tensor(mA.element_type, sA_layout, 16)
        sB = smem.allocate_tensor(mB.element_type, sB_layout, 16)
        thr_copy_A = tiled_copy_A.get_slice(tidx)
        thr_copy_B = tiled_copy_B.get_slice(tidx)
        tAgA = thr_copy_A.partition_S(gA); tAsA = thr_copy_A.partition_D(sA)
        tBgB = thr_copy_B.partition_S(gB); tBsB = thr_copy_B.partition_D(sB)
        tCsA = thr_mma.partition_A(sA); tCsB = thr_mma.partition_B(sB)
        tCgC = thr_mma.partition_C(gC)
        tCrA = tiled_mma.make_fragment_A(tCsA)
        tCrB = tiled_mma.make_fragment_B(tCsB)
        tCrC = tiled_mma.make_fragment_C(tCgC)
        tCrC.fill(0.0)

        # (C) mainloop
        k_tile_count = cute.size(tAgA, mode=[3])
        k_block_max  = cute.size(tCrA, mode=[2])
        for k_tile in range(k_tile_count):
            cute.copy(tiled_copy_A, tAgA[None, None, None, k_tile], tAsA[None, None, None])
            cute.copy(tiled_copy_B, tBgB[None, None, None, k_tile], tBsB[None, None, None])
            cute.arch.cp_async_commit_group()
            cute.arch.cp_async_wait_group(0)
            cute.arch.barrier()
            for k_block in cutlass.range_constexpr(k_block_max):
                cute.autovec_copy(tCsA[None, None, k_block], tCrA[None, None, k_block])
                cute.autovec_copy(tCsB[None, None, k_block], tCrB[None, None, k_block])
                cute.gemm(tiled_mma, tCrC,
                          tCrA[None, None, k_block], tCrB[None, None, k_block], tCrC)
            cute.arch.barrier()

        atom_store = cute.make_copy_atom(cute.nvgpu.CopyUniversalOp(), mC.element_type)
        cute.copy(atom_store, tCrC, tCgC)


M, N, K = 256, 256, 64
a = torch.empty(K, M, dtype=torch.float32).normal_().permute(1, 0).cuda()
b = torch.empty(K, N, dtype=torch.float32).normal_().permute(1, 0).cuda()
c = torch.empty(N, M, dtype=torch.float32).zero_().permute(1, 0).cuda()
SimpleSGemm()(from_dlpack(a, assumed_align=16),
              from_dlpack(b, assumed_align=16),
              from_dlpack(c, assumed_align=16))
torch.cuda.synchronize()
ref = torch.einsum("mk,nk->mn", a, b)
torch.testing.assert_close(c.cpu(), ref.cpu(), atol=1e-3, rtol=1e-4)
print("OK")
